# 15- The Unleashed Hybrid Pipeline (Ablation Test)

**Goal:** Establish the theoretical maximum score on the MedQA benchmark by allowing the LLM to use its internal parametric memory.
This pipeline tests the exact same Hybrid Retrieval architecture but REMOVES the strict anti-hallucination guardrails to observe the impact on Ragas Faithfulness vs. MedQA Accuracy:
1. **Query Generation:** English -> German Translation + German HyDE Passage + German UMLS Keywords.
2. **Hybrid Retrieval:** Dense Search (PGVector) + Sparse Search (BM25) merged via Reciprocal Rank Fusion (RRF).
3. **Reranking:** Cross-Encoder isolates the Top 10 chunks.
4. **Unleashed Generation:** The LLM is explicitly permitted to use its general medical knowledge if the retrieved context is insufficient (The `NOT_IN_CORPUS` rule is disabled).

## Installs

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests spacy rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 143.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 

## Vertex Patch

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## Setup DB, Reranker & Build Local BM25

In [ ]:
import os, json, time, random
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30})

# 1. Setup Dense Vector Store
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name="awmf_baseline_bge", connection=engine, use_jsonb=True)
dense_retriever = vs.as_retriever(search_kwargs={"k": 30})

# 2. Setup Sparse BM25 Store (Extract all texts from DB)
print("Loading texts for BM25 Index...")
with engine.connect() as conn:
    res = conn.execute(text("SELECT document FROM langchain_pg_embedding WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = 'awmf_baseline_bge')"))
    all_texts = [row[0] for row in res]

tokenized_corpus = [doc.lower().split(" ") for doc in all_texts]
bm25_retriever = BM25Okapi(tokenized_corpus)
print(f"BM25 Index built with {len(all_texts)} chunks.")

# 3. Setup Reranker (Mandatory for 0.9+ scores)
print("Loading Reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# The Ultimate Query Generator (Translation + HyDE combined)
query_gen_prompt = PromptTemplate(template="""You are a German medical expert.
1. First, precisely TRANSLATE the English question into German.
2. Second, write a 2-sentence hypothetical answer to the question in formal German clinical guideline terminology.
Do NOT use bullet points. Output ONLY the German translation followed directly by the German hypothetical answer.

English Question:
{question}""", input_variables=["question"])

# The "Unleashed" Prompt (Allows Parametric Memory)
qa_prompt = PromptTemplate(template="""You are an expert medical AI.
Read the German clinical guidelines below to answer the medical question in ENGLISH.

INSTRUCTIONS:
1. Try to base your response on the provided German context.
2. If the context does NOT contain enough information, use your comprehensive medical training to answer the question correctly.
3. Do NOT say the information is missing or not in the corpus. Just provide the best possible medical answer.

Context (German):
{context}

Question (English):
{question}

Answer (English):""", input_variables=["context","question"])
print("Setup complete.")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Setup complete.


## UMLS Ontology

In [5]:
import requests, functools, spacy

try: nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as _dl; _dl("en_core_web_sm"); nlp = spacy.load("en_core_web_sm")

UMLS_API_KEY = userdata.get('UMLS_API_KEY')
UTS = "https://uts-ws.nlm.nih.gov/rest"
GENERIC = {"patient","patients","treatment","diagnosis","management","therapy","risk","cause", "symptom","disease","condition","prognosis"}

def candidate_phrases(question, max_n=6):
    doc = nlp(question)
    out, seen = [], set()
    for ch in doc.noun_chunks:
        t = " ".join(w.text for w in ch if not w.is_stop and not w.is_punct).strip()
        tl = t.lower()
        if len(tl) < 3 or tl in GENERIC or tl in seen: continue
        seen.add(tl); out.append(t)
        if len(out) >= max_n: break
    return out

@functools.lru_cache(maxsize=4096)
def search_cui(term):
    try:
        r = requests.get(f"{UTS}/search/current", params={"string": term, "apiKey": UMLS_API_KEY, "pageSize": 1, "searchType": "words"}, timeout=20)
        hits = r.json().get("result", {}).get("results", []) if r.ok else []
        if hits and hits[0].get("ui") not in (None, "NONE"): return hits[0]["ui"], hits[0].get("name", term)
    except: pass
    return None

@functools.lru_cache(maxsize=4096)
def german_atoms(cui):
    try:
        r = requests.get(f"{UTS}/content/current/CUI/{cui}/atoms", params={"language": "GER", "apiKey": UMLS_API_KEY, "pageSize": 50}, timeout=20)
        if not r.ok: return []
        names = []
        for a in r.json().get("result", []):
            n = (a.get("name") or "").strip()
            if n and all(n.lower() != x.lower() for x in names): names.append(n)
        return names[:4]
    except: return []

def get_umls_keywords(question):
    de_terms = []
    for ph in candidate_phrases(question):
        hit = search_cui(ph)
        if hit:
            de = german_atoms(hit[0])
            de_terms.extend(de)
    return " ".join(list(dict.fromkeys(de_terms))) # Deduplicate and join

## Hybrid Search & Reciprocal Rank Fusion

In [6]:
def retrieve_hybrid(question, k_final=10):
    # 1. Generate Translation & HyDE
    hyde_translation = safe_invoke(mistral, query_gen_prompt.format(question=question))

    # 2. Get UMLS Keywords
    umls_keywords = get_umls_keywords(question)

    # 3. Combine into Ultimate Search Query
    hybrid_query = f"{hyde_translation} {umls_keywords}".strip()

    # 4. Dense Search (PGVector)
    dense_docs = [d.page_content for d in dense_retriever.invoke(hybrid_query)]

    # 5. Sparse Search (BM25)
    tokenized_query = hybrid_query.lower().split(" ")
    sparse_docs = bm25_retriever.get_top_n(tokenized_query, all_texts, n=30)

    # 6. Reciprocal Rank Fusion (RRF)
    rrf_scores = {}
    for rank, doc in enumerate(dense_docs):
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 1.0 / (60 + rank)
    for rank, doc in enumerate(sparse_docs):
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 1.0 / (60 + rank)

    # Sort by RRF score and take top 30 unique chunks
    fused_docs = [doc for doc, score in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:30]]

    # 7. Cross-Encoder Reranking
    scores = reranker.predict([[hybrid_query, t] for t in fused_docs])
    reranked_texts = [t for _, t in sorted(zip(scores, fused_docs), key=lambda x: x[0], reverse=True)]

    return reranked_texts[:k_final]

# Smoke Test
_q = df.iloc[0]['English_Open_Question']
print("Testing Hybrid Pipeline...")
print("Top Document:", retrieve_hybrid(_q)[0][:200], "...")

Testing Hybrid Pipeline...
Top Document: NVL Asthma 
Langfassung – Version 5.0 
 © NVL-Programm 2024      Seite 116 
7.4 Initiale Versorgung des Asthmaanfalls: Arztpraxis oder Notaufnahme 
Abbildung 7:Initiale Versorgung des Asthmaanfalls:  ...


## The Main Generation Loop

In [7]:
# Load corpus-grounded GT (built in notebook 11)
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()

work = df.reset_index(drop=True) # Full 200 Questions
print(f"Generating Ultimate Hybrid answers for {len(work)} questions")

out_file = DRIVE_PATH + "HYBRID_UNLEASHED_results.json"
if os.path.exists(out_file):
    res = json.load(open(out_file)); done = set(res["question"])
    print(f"  resuming -- {len(done)} already done")
else:
    res = {"question":[],"answer":[],"contexts":[],"medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()

for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx = retrieve_hybrid(q)
        ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))

        # Format alignment
        res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
        done.add(q)
        json.dump(res, open(out_file,"w"))

        if len(res["question"]) % 10 == 0: print(f"  [hybrid] {len(res['question'])}/{len(work)}")
        time.sleep(1.5)
    except Exception as e:
        print("err", i, str(e)[:100]); time.sleep(8)

print(f"Hybrid Generation complete -> {out_file}")

Generating Ultimate Hybrid answers for 200 questions
  [hybrid] 10/200
  [hybrid] 20/200
  [hybrid] 30/200
  [hybrid] 40/200
  [hybrid] 50/200
  [hybrid] 60/200
  [hybrid] 70/200
  [hybrid] 80/200
  [hybrid] 90/200
  [hybrid] 100/200
  [hybrid] 110/200
  [hybrid] 120/200
  [hybrid] 130/200
  [hybrid] 140/200
  [hybrid] 150/200
  [hybrid] 160/200
  [hybrid] 170/200
  [hybrid] 180/200
  [hybrid] 190/200
  [hybrid] 200/200
Hybrid Generation complete -> /content/drive/MyDrive/HYBRID_UNLEASHED_results.json


## Final Evaluation

In [8]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score_unleashed(path, gt_key):
    d = json.load(open(path))
    dd = {"question":d["question"], "answer":d["answer"],
          "contexts":d["contexts"], "ground_truth":d[gt_key]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return dict(out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3))

print("=== UNLEASHED HYBRID SCORES ===")
print("FULL 200 (MedQA Ground Truth):", score_unleashed(out_file, "medqa_ground_truth"))

/tmp/ipykernel_1188/3637763638.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_1188/3637763638.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_1188/3637763638.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precis

=== UNLEASHED HYBRID SCORES ===


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

FULL 200 (MedQA Ground Truth): {'context_precision': np.float64(0.152), 'context_recall': np.float64(0.247), 'faithfulness': np.float64(0.46), 'answer_relevancy': np.float64(0.58)}
